# 04 — Data Quality, Cleaning & Outlier Profiling

**Objective:** Audit data cleanliness without destructive data loss.
- Verify missing values (confirmed 0).
- Profile statistical outliers (Tukey IQR and 3-sigma z-scores).
- Analyze duplicate operational records.
- Produce cleaned master dataset (`cleaned_master.parquet`) and audit log.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import load_raw_yearly_data
from src.data.cleaning import DataCleaner


## 1. Non-Destructive Cleaning Pipeline
Executing `DataCleaner` with non-destructive outlier profiling and duplicate retention to preserve high-frequency turbine dynamics.


In [2]:
raw_df = load_raw_yearly_data(ROOT / "data" / "raw")
cleaner = DataCleaner(drop_duplicates=False, handle_outliers="flag")

cleaned_df, cleaning_log = cleaner.fit_transform(raw_df)

print(f"Initial row count: {cleaning_log['initial_rows']}")
print(f"Final row count:   {cleaning_log['final_rows']}")
print("Operations performed:")
for op in cleaning_log["operations"]:
    print(f"  - {op}")

out_log = ROOT / "outputs" / "reports" / "cleaning_log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(cleaning_log, f, indent=2)
print(f"Saved cleaning log to {out_log.relative_to(ROOT)}")


Initial row count: 36733
Final row count:   36733
Operations performed:
  - Zero missing values verified across all columns.
  - Identified 7 duplicate records (retained to preserve natural sampling frequency).
  - Outliers flagged and retained without destructive clipping.
Saved cleaning log to outputs\reports\cleaning_log.json


## 2. Exporting Cleaned Master Dataset
Serializing master dataset to Parquet for fast, lossless I/O.


In [3]:
out_parquet = ROOT / "data" / "processed" / "cleaned_master.parquet"
out_parquet.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_parquet(out_parquet, index=False)
print(f"Saved cleaned master dataset to {out_parquet.relative_to(ROOT)} ({out_parquet.stat().st_size:,} bytes)")
print("Data quality and cleaning pipeline COMPLETE.")


Saved cleaned master dataset to data\processed\cleaned_master.parquet (1,489,991 bytes)
Data quality and cleaning pipeline COMPLETE.
